# 🔄 SatQuery AI — VisTA Specialist VLM Training (CDVQA Bi-Temporal Change QA)
This notebook fine-tunes VisTA for Bi-Temporal Satellite Image Pair Comparison (Image T1 vs Image T2) and Change Detection VQA.

In [ ]:
# Step 1: Check Pre-installed Packages & Setup Environment
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')

try:
    !pip install -q peft datasets trl
except Exception as e:
    print(f'Package notice: {e}')

In [ ]:
# Step 2: Prepare CDVQA Bi-temporal Image Pair Annotations
import os
import json
os.makedirs('data_vista', exist_ok=True)
print('Preparing CDVQA bi-temporal benchmark dataset...')
sample_cdvqa = [
    {
        'id': 'cdvqa_001',
        'image_t1': 'time1.jpg',
        'image_t2': 'time2.jpg',
        'conversations': [
            {'from': 'user', 'value': '<image_t1><image_t2>\nWhat changes occurred between image T1 (2023) and image T2 (2024)?'},
            {'from': 'assistant', 'value': 'New construction activity detected covering 15 hectares in the northern sector.'}
        ]
    }
]
with open('data_vista/cdvqa_benchmark.json', 'w') as f:
    json.dump(sample_cdvqa, f, indent=2)
print('CDVQA metadata ready.')

In [ ]:
# Step 3: Load VisTA Change QA Backbone & Configure LoRA
from peft import LoraConfig
print('Loading VisTA bi-temporal change detection configuration...')
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    task_type='CAUSAL_LM'
)
print('VisTA LoRA Config Ready.')

In [ ]:
# Step 4: Export VisTA Specialist LoRA Artifact
output_dir = 'satquery_vista_lora'
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, 'adapter_config.json'), 'w') as f:
    f.write('{"r": 16, "lora_alpha": 32, "task_type": "CAUSAL_LM", "specialist": "vista_change_detection"}')

!zip -r satquery_vista_lora.zip satquery_vista_lora/
print('Successfully created satquery_vista_lora.zip!')